#### **Import Libraries**


In [123]:
from __future__ import annotations

import os
from typing import Optional

import albumentations
import cv2
import numpy as np
import wandb
import pytorch_lightning as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from albumentations.pytorch import ToTensorV2
from e2cnn import gspaces
from e2cnn import nn as e2nn
from PIL import Image
from pytorch_lightning.callbacks import LearningRateMonitor
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader
from torchmetrics.classification import (
    Accuracy,
    ConfusionMatrix,
    F1Score,
    Precision,
    Recall,
)
from torchvision import datasets
from sklearn.model_selection import train_test_split

#### **Set Hyperparameters**


In [124]:
image_size = 151
kernel_size = 5
N = 16
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model_path = '/media/drive1/sazzat/rgc/byol_on_rtx/j2fa39p6/checkpoints/epoch=499-step=115000.ckpt'
data_dir = '/media/drive1/sazzat/rgc/dataset/no_spurious_masked_png'
mean = [0.0032]
std = [0.0376]

In [125]:
def _grayscale_loader(path: str) -> Image.Image:
    with open(path, "rb") as f:
        img = Image.open(f)
        return img.convert("L")


class Bent(datasets.ImageFolder):
    """ImageFolder subclass that loads images as single-channel grayscale."""

    def __init__(
        self,
        root: str,
        transform: Optional[torchvision.transforms.Compose] = None,
        target_transform: Optional[torchvision.transforms.Compose] = None,
    ) -> None:
        super().__init__(
            root=os.path.expanduser(root),
            transform=transform,
            target_transform=target_transform,
            loader=_grayscale_loader,
        )

    def __repr__(self) -> str:
        fmt_str = "Dataset " + self.__class__.__name__ + "\n"
        fmt_str += f"    Number of datapoints: {self.__len__()}\n"
        fmt_str += f"    Root Location: {self.root}\n"
        return fmt_str

In [126]:
class Transforms:
    def __init__(self, transforms: albumentations.Compose):
        self.transforms = transforms

    def __call__(self, img, *args, **kwargs):
        return self.transforms(image=np.array(img))["image"]


class GalaxyDataset(pl.LightningDataModule):
    def __init__(
        self,
        data_dir,
        batch_size=32,
        num_workers=4,
        transform=None,
        test_transform=None,
        train_indices=None,
        test_indices=None,
    ):
        super(GalaxyDataset, self).__init__()
        self.data_dir = data_dir
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.transform = transform
        self.test_transform = test_transform
        self.train_indices = train_indices
        self.test_indices = test_indices

    def setup(self, stage=None):
        entire_dataset_train = Bent(
            root=self.data_dir,
            transform=self.transform,
            target_transform=None,
        )

        entire_dataset_test = Bent(
            root=self.data_dir,
            transform=self.transform,
            target_transform=None,
        )

        self.train_dataset = torch.utils.data.Subset(entire_dataset_train, self.train_indices)
        self.test_dataset = torch.utils.data.Subset(entire_dataset_test, self.test_indices)

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            num_workers=self.num_workers,
            shuffle=True,
        )

    def val_dataloader(self):
        return DataLoader(
            self.test_dataset,
            batch_size=self.batch_size,
            num_workers=self.num_workers,
            shuffle=False,
        )

    def test_dataloader(self):
        return DataLoader(
            self.test_dataset,
            batch_size=self.batch_size,
            num_workers=self.num_workers,
            shuffle=False,
        )

In [127]:
PROJECT_ROOT = "/media/drive1/sazzat/rgc"
BYOL_CHECKPOINT = os.path.join(PROJECT_ROOT, "byol", "best.pt")
DATA_DIR = os.path.join(PROJECT_ROOT, "dataset", "no_spurious_masked_png")

# Derive CLASS_NAMES from dataset so index order always matches ImageFolder (alphabetical by folder name)
_temp_bent = Bent(root=DATA_DIR)
CLASS_NAMES = _temp_bent.classes
NUM_CLASSES = len(CLASS_NAMES)
_samples_per_class = [sum(1 for _, j in _temp_bent.samples if j == i) for i in range(NUM_CLASSES)]
print("Class index -> name (from dataset):", dict(enumerate(CLASS_NAMES)))
print("Samples per class (index order):", dict(zip(CLASS_NAMES, _samples_per_class)))

Class index -> name (from dataset): {0: 'nat', 1: 'sfri', 2: 'sfrii', 3: 'wat'}
Samples per class (index order): {'nat': 475, 'sfri': 322, 'sfrii': 588, 'wat': 675}


In [128]:
class DSteerableLeNet(nn.Module):
    """Steerable CNN for image classification."""

    def __init__(
        self,
        imsize: int = 151,
        kernel_size: int = 5,
        N: int = 16
    ) -> None:
        """
        Initialize the network.

        :param imsize: size of the input image
        :type imsize: int
        :param kernel_size: size of the convolutional kernel
        :type kernel_size: int
        :param N: number of rotations
        :type N: int
        """
        super(DSteerableLeNet, self).__init__()
        self.imsize = imsize
        self.kernel_size = kernel_size
        self.N = N

        z = 0.5 * (self.imsize - 2)
        z = int(0.5 * (z - 2))
        self._feat_size = 16 * z * z

        self.r2_act = gspaces.FlipRot2dOnR2(self.N)

        in_type = e2nn.FieldType(self.r2_act, [self.r2_act.trivial_repr])
        self.input_type = in_type

        out_type = e2nn.FieldType(self.r2_act, 6 * [self.r2_act.regular_repr])
        self.mask = e2nn.MaskModule(in_type, self.imsize, margin=1)
        self.conv1 = e2nn.R2Conv(
            in_type,
            out_type,
            kernel_size=self.kernel_size,
            padding=1,
            bias=False
        )
        self.relu1 = e2nn.ReLU(out_type, inplace=True)
        self.pool1 = e2nn.PointwiseMaxPoolAntialiased(out_type, kernel_size=2)
        self.drop1 = e2nn.PointwiseDropout(out_type, p=0.5)

        in_type = self.pool1.out_type
        out_type = e2nn.FieldType(self.r2_act, 16 * [self.r2_act.regular_repr])
        self.conv2 = e2nn.R2Conv(
            in_type,
            out_type,
            kernel_size=self.kernel_size,
            padding=1,
            bias=False
        )
        self.relu2 = e2nn.ReLU(out_type, inplace=True)
        self.pool2 = e2nn.PointwiseMaxPoolAntialiased(out_type, kernel_size=2)
        self.drop2 = e2nn.PointwiseDropout(out_type, p=0.5)

        self.gpool = e2nn.GroupPooling(out_type)

        self.fc = nn.Linear(self._feat_size, 2048)

        # dummy parameter for tracking device
        self.dummy = nn.Parameter(torch.empty(0))

        self.gradients = None

    @property
    def feature_dim(self) -> int:
        return self._feat_size

    def activations_hook(self, grad: torch.Tensor) -> None:
        """
        Track the gradient of the network.

        :param grad: gradient tensor
        :type grad: torch.Tensor
        """
        self.gradients = grad

    def forward_features(self, x: torch.Tensor) -> torch.Tensor:
        """Forward up to (and including) gpool; used by BYOLDownstreamClassifier and Grad-CAM."""
        x = e2nn.GeometricTensor(x, self.input_type)

        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)
        x = self.drop1(x)

        x = self.conv2(x)
        x = self.relu2(x)
        _ = x.tensor.register_hook(self.activations_hook)
        x = self.pool2(x)
        x = self.drop2(x)

        x = self.gpool(x)
        x = x.tensor
        x = x.view(x.size()[0], -1)
        return x

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass of the network.

        :param x: input tensor
        :type x: torch.Tensor
        """
        x = self.forward_features(x)
        x = self.fc(x)
        return x

    def get_activations_gradient(self) -> torch.Tensor:
        """
        Get the gradient of the network.

        :return: gradient tensor
        :rtype: torch.Tensor
        """
        return self.gradients

    def get_activations(self, x: torch.Tensor) -> torch.Tensor:
        """
        Get the activations of the network.

        :param x: input tensor
        :type x: torch.Tensor
        :return: activations tensor
        :rtype: torch.Tensor
        """
        x = e2nn.GeometricTensor(x, self.input_type)

        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)
        x = self.drop1(x)

        x = self.conv2(x)
        x = self.relu2(x)

        x = x.tensor

        return x

In [129]:
class BYOLDownstreamClassifier(pl.LightningModule):
    """BYOL-pretrained DSteerableLeNet + classification head, wrapped in Lightning."""

    def __init__(
        self,
        num_classes: int = 4,
        learning_rate: float = 3e-4,
        weight_decay: float = 1e-6,
        pretrained_path: str | None = None,
        freeze_encoder: bool = False,
        image_size: int = 151,
        kernel_size: int = 5,
        N: int = 16,
    ):
        super().__init__()
        self.save_hyperparameters()

        # ---- encoder ----
        self.encoder = DSteerableLeNet(imsize=image_size, kernel_size=kernel_size, N=N)

        if pretrained_path is not None:
            ckpt = torch.load(pretrained_path, map_location="cpu")
            self.encoder.load_state_dict(ckpt["model_state_dict"])
            print(f"Loaded BYOL weights from {pretrained_path}")

        if freeze_encoder:
            for p in self.encoder.parameters():
                p.requires_grad = False

        # ---- classification head (replaces the BYOL fc layer) ----
        feat_dim = self.encoder.feature_dim
        self.classifier = nn.Sequential(
            nn.Linear(feat_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(256, num_classes),
        )

        self.criterion = nn.CrossEntropyLoss()

        # ---- metrics ----
        self.train_accuracy = Accuracy(task="multiclass", num_classes=num_classes)
        self.val_accuracy = Accuracy(task="multiclass", num_classes=num_classes)
        self.test_accuracy = Accuracy(task="multiclass", num_classes=num_classes)
        self.precision_metric = Precision(task="multiclass", num_classes=num_classes)
        self.recall_metric = Recall(task="multiclass", num_classes=num_classes)
        self.f1_metric = F1Score(task="multiclass", num_classes=num_classes)
        self.confusion_matrix_metric = ConfusionMatrix(task="multiclass", num_classes=num_classes)

        self.test_outputs: list[dict] = []

    def forward(self, x):
        features = self.encoder.forward_features(x)
        return self.classifier(features)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        acc = self.train_accuracy(y_hat, y)
        self.log("train/loss", loss, on_step=True, on_epoch=True, prog_bar=True, logger=True)
        self.log("train/acc", acc, on_step=True, on_epoch=True, prog_bar=True, logger=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        acc = self.val_accuracy(y_hat, y)
        prec = self.precision_metric(y_hat, y)
        rec = self.recall_metric(y_hat, y)
        f1 = self.f1_metric(y_hat, y)
        self.log("val/loss", loss, on_epoch=True, logger=True)
        self.log("val/acc", acc, on_epoch=True, prog_bar=True, logger=True)
        self.log("val/precision", prec, on_epoch=True, logger=True)
        self.log("val/recall", rec, on_epoch=True, logger=True)
        self.log("val/f1_score", f1, on_epoch=True, logger=True)
        return loss

    def test_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        self.test_outputs.append({"y_hat": y_hat, "y": y})
        return loss

    def on_test_epoch_end(self):
        y_hats = torch.cat([o["y_hat"] for o in self.test_outputs], dim=0)
        y_true = torch.cat([o["y"] for o in self.test_outputs], dim=0)

        acc = self.test_accuracy(y_hats, y_true)
        precision = self.precision_metric(y_hats, y_true)
        recall = self.recall_metric(y_hats, y_true)
        f1 = self.f1_metric(y_hats, y_true)

        self.log("test/acc", acc, on_epoch=True, logger=True)
        self.log("test/precision", precision, on_epoch=True, logger=True)
        self.log("test/recall", recall, on_epoch=True, logger=True)
        self.log("test/f1_score", f1, on_epoch=True, logger=True)

        y_true_np = y_true.cpu().numpy()
        y_pred_np = torch.argmax(y_hats, dim=1).cpu().numpy()

        report = classification_report(
            y_true_np, y_pred_np, target_names=CLASS_NAMES, zero_division=0,
        )
        self.logger.experiment.log(
            {"classification_report": wandb.Html(f"<pre>{report}</pre>")}
        )
        self.logger.experiment.log(
            {
                "confusion_matrix": wandb.plot.confusion_matrix(
                    probs=None,
                    y_true=y_true_np,
                    preds=y_pred_np,
                    class_names=CLASS_NAMES,
                )
            }
        )

        self.test_outputs.clear()

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(
            self.parameters(),
            lr=self.hparams.learning_rate,
            weight_decay=self.hparams.weight_decay,
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, factor=0.5, patience=5, mode="min",
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "monitor": "val/loss"},
        }

In [130]:
hparams = {
    "model_name": "byol_dsteerable_downstream",
    "learning_rate": 0.0009,
    "weight_decay": 2.0802390891987617e-05,
    "batch_size": 32,
    "num_workers": 4,
    "max_epochs": 500,
    "freeze_encoder": False,
    "image_size": 151,
    "train_transform" : torchvision.transforms.Compose(
        [
            torchvision.transforms.Resize((150, 150)),
            torchvision.transforms.ToTensor(),
            torchvision.transforms.Pad(padding=1, fill=0, padding_mode="constant"),
            torchvision.transforms.Normalize(
                mean=[0.0032],
                std=[0.0376],
            ),
        ]
    ),
    "test_transform": torchvision.transforms.Compose(
        [
            torchvision.transforms.Resize((150, 150)),
            torchvision.transforms.ToTensor(),
            torchvision.transforms.Pad(padding=1, fill=0, padding_mode="constant"),
            torchvision.transforms.Normalize(
                mean=[0.0032],
                std=[0.0376],
            ),
            torchvision.transforms.RandomRotation(
                degrees=360,
                interpolation=torchvision.transforms.InterpolationMode.BILINEAR,
                expand=False
            ),
        ]
    )

}



model = BYOLDownstreamClassifier(
    num_classes=NUM_CLASSES,
    learning_rate=hparams["learning_rate"],
    weight_decay=hparams["weight_decay"],
    pretrained_path=BYOL_CHECKPOINT,
    freeze_encoder=hparams["freeze_encoder"],
    image_size=hparams["image_size"],
)




Loaded BYOL weights from /media/drive1/sazzat/rgc/byol/best.pt


#### **Load Pretrained Model**


In [131]:
# Load fine-tuned checkpoint for Grad-CAM (optional; use this so CAM uses your trained weights)
# If you use the Lightning checkpoint path (e.g. model_path), uncomment and run:
# model = BYOLDownstreamClassifier.load_from_checkpoint(
#     model_path,
#     num_classes=NUM_CLASSES,
#     learning_rate=hparams["learning_rate"],
#     weight_decay=hparams["weight_decay"],
#     pretrained_path=BYOL_CHECKPOINT,
#     freeze_encoder=hparams["freeze_encoder"],
#     image_size=hparams["image_size"],
# )
# model = model.to(device)
# model.eval()

#### **Prepare Data**


In [132]:
entire_dataset = Bent(root=DATA_DIR, transform=hparams["train_transform"])

train_indices, test_indices = train_test_split(
    range(len(entire_dataset)),
    test_size=0.2,
    stratify=[entire_dataset.samples[i][1] for i in range(len(entire_dataset))],
    random_state=42,
)

data_module = GalaxyDataset(
    data_dir=DATA_DIR,
    batch_size=hparams["batch_size"],
    num_workers=hparams["num_workers"],
    transform=hparams["train_transform"],
    test_transform=hparams["test_transform"],
    train_indices=train_indices,
    test_indices=test_indices,
)
data_module.setup()

# Dataset that yields (image, target, filename, path) for Grad-CAM loop
class BentWithFilename(Bent):
    def __getitem__(self, index):
        img, target = super().__getitem__(index)
        path = self.samples[index][0]
        filename = os.path.basename(path)
        return img, target, filename, path

dataset = BentWithFilename(root=DATA_DIR, transform=hparams["test_transform"])



#### **Check Model Accuracy**


In [133]:
# labels = [0 if filename.split(
#     '_')[0] == '100' else 1 for filename in filenames]
# print(labels)

# # check accuracy by comparing the predicted label to the true label
# preds = []

# for image in images:
#     image = image.unsqueeze(0)
#     output = model(image)
#     pred = F.softmax(output, dim=1).argmax(dim=1, keepdim=True)
#     preds.append(pred.item())

# print(preds)

#### **Implement Grad-CAM**


In [134]:
def get_cam(
    model: pl.LightningModule,
    image: torch.Tensor,
    target: torch.Tensor,
    CONTOUR_THRESHOLD: float = 0.6
) -> tuple:
    """
    Get the Grad-CAM heatmap using the model's encoder.
    :param model: BYOLDownstreamClassifier
    :param image: Input image tensor (1, C, H, W).
    :param target: Target class index tensor.
    :param CONTOUR_THRESHOLD: contour activation threshold (0–1)
    :return: (heatmap BGR, contours, pred class index).
    """

    encoder = model.encoder
    encoder.train()

    image = image.to(device)
    target = target.to(device).long()

    if target.dim() == 0:
        target = target.unsqueeze(0)

    model.zero_grad()

    output = model(image)

    pred = torch.argmax(F.softmax(output, dim=1), dim=1).detach().cpu()

    # Backprop the score of the target class
    one_hot = torch.zeros_like(output)
    one_hot[0, target[0]] = 1.0
    (output * one_hot).sum().backward()

    gradient = encoder.get_activations_gradient()

    if gradient is None:
        raise RuntimeError(
            "No gradients captured; ensure encoder.forward_features registers the hook."
        )

    pooled_gradient = torch.mean(gradient, dim=[0, 2, 3])

    activations = encoder.get_activations(image).detach()

    for i in range(activations.shape[1]):
        activations[:, i, :, :] *= pooled_gradient[i]

    heatmap = torch.mean(activations, dim=1).squeeze()

    heatmap = torch.relu(heatmap)

    heatmap = heatmap.cpu().numpy()

    if heatmap.max() > 0:
        heatmap = heatmap / heatmap.max()

    # Resize once; keep 2D float (0–1) for matplotlib-style level contours
    heatmap_2d = cv2.resize(heatmap, (150, 150))
    heatmap_uint8 = np.uint8(255 * heatmap_2d)

    # -------- OpenCV binary contours (one threshold) --------
    threshold_value = int(CONTOUR_THRESHOLD * 255)
    _, thresh = cv2.threshold(
        heatmap_uint8, threshold_value, 255, cv2.THRESH_BINARY
    )
    cnt_result = cv2.findContours(
        thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    contours = cnt_result[0] if len(cnt_result) == 2 else cnt_result[1]

    heatmap_bgr = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)

    return heatmap_bgr, contours, pred, heatmap_2d

#### **Create and Save CAM Images**


In [135]:
GRAD_CAM_DIR = os.path.join(PROJECT_ROOT, "grad_cam_without_sp")
for sub in ("heatmap", "result", "image", "contour", "contour_levels"):
    os.makedirs(os.path.join(GRAD_CAM_DIR, sub), exist_ok=True)

model = model.to(device)
model.eval()

preds = []

for i, data in enumerate(dataset):
    image, target, filename, path = data

    target_t = torch.tensor([target]).to(device)
    image_batch = image.unsqueeze(0)

    heatmap, contours, pred, heatmap_2d = get_cam(
        model,
        image_batch,
        target_t,
        CONTOUR_THRESHOLD=0.1
    )
    cv2.imwrite(os.path.join(GRAD_CAM_DIR, "heatmap", filename), heatmap)

    # Overlay: use same image as input (path), or set overlay_dir for a different folder
    overlay_dir = data_dir + 'shahal_good/' if os.path.exists(data_dir + 'shahal_good/') else os.path.dirname(path) + os.sep
    image_display = Image.open(overlay_dir + filename) if os.path.exists(overlay_dir + filename) else Image.open(path)
    image_np = np.array(image_display)
    if image_np.ndim == 2:
        image_np = cv2.cvtColor(image_np, cv2.COLOR_GRAY2RGB)
    else:
        image_np = cv2.cvtColor(image_np, cv2.COLOR_RGB2BGR)
    # Resize to match heatmap (150, 150) so blend and contours align
    image_np = cv2.resize(image_np, (heatmap.shape[1], heatmap.shape[0]), interpolation=cv2.INTER_LINEAR)

    result = heatmap * 0.3 + image_np * 0.5
    cv2.imwrite(os.path.join(GRAD_CAM_DIR, "result", filename), result)
    cv2.imwrite(os.path.join(GRAD_CAM_DIR, "image", filename), image_np)

    # OpenCV: draw binary contours (one threshold) on original image
    cv2.drawContours(image_np, contours, -1, (255, 255, 255), thickness=1)
    cv2.imwrite(os.path.join(GRAD_CAM_DIR, "contour", filename), image_np)

    # Matplotlib-style: level contours (multiple smooth iso-lines) on image
    fig, ax = plt.subplots(figsize=(1.5, 1.5), dpi=100)
    image_rgb = cv2.cvtColor(image_np, cv2.COLOR_BGR2RGB)
    ax.imshow(image_rgb)
    levels = np.linspace(heatmap_2d.min(), heatmap_2d.max(), 5)
    ax.contour(heatmap_2d, levels=levels, colors="k", alpha=0.5, linewidths=0.5)
    ax.axis("off")
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    plt.savefig(os.path.join(GRAD_CAM_DIR, "contour_levels", filename))
    plt.close(fig)

    preds.append([pred.squeeze().item(), target, filename])

# save the predictions to csv: pred (0-3), ground (0-3), filename, pred_name, ground_name
with open(os.path.join(GRAD_CAM_DIR, "predictions.csv"), "w") as f:
    f.write("pred,ground,filename,pred_name,ground_name\n")
    for pred_idx, ground_idx, fname in preds:
        f.write(f"{pred_idx},{ground_idx},{fname},{CLASS_NAMES[pred_idx]},{CLASS_NAMES[ground_idx]}\n")

#### **Class index → name mapping & accuracy**

Class indices **0, 1, 2, 3** come from **`torchvision.datasets.ImageFolder`**: it assigns indices in **alphabetical order** of the subdirectory names under `DATA_DIR` (`dataset/no_spurious_masked_png`). The notebook sets **`CLASS_NAMES = ["nat", "sfri", "sfrii", "wat"]`** in that same order so that **index ↔ name** is:

| Index | Name  |
|-------|-------|
| 0     | nat   |
| 1     | sfri  |
| 2     | sfrii |
| 3     | wat   |

You can confirm at runtime with `Bent(root=DATA_DIR).class_to_idx` (alphabetical by folder name).

In [136]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

csv_path = os.path.join(GRAD_CAM_DIR, "predictions.csv")
df = pd.read_csv(csv_path)

# CSV must have pred, ground, filename (from the Grad-CAM cell above)
if "ground" not in df.columns:
    raise ValueError("predictions.csv missing 'ground' column. Re-run the Grad-CAM cell to save pred,ground,filename.")
y_true = df["ground"].values
y_pred = df["pred"].values

acc = accuracy_score(y_true, y_pred)
print(f"Accuracy (Grad-CAM run): {acc:.4f} ({acc * 100:.2f}%)")
print("\nClassification report (class names = CLASS_NAMES):")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))
print("Confusion matrix (rows=true, cols=pred):")
print(confusion_matrix(y_true, y_pred))

Accuracy (Grad-CAM run): 0.2034 (20.34%)

Classification report (class names = CLASS_NAMES):
              precision    recall  f1-score   support

         nat       0.29      0.25      0.27       475
        sfri       0.17      0.85      0.29       322
       sfrii       0.44      0.04      0.07       588
         wat       0.00      0.00      0.00       675

    accuracy                           0.20      2060
   macro avg       0.22      0.29      0.16      2060
weighted avg       0.22      0.20      0.13      2060

Confusion matrix (rows=true, cols=pred):
[[120 347   8   0]
 [ 37 275  10   0]
 [ 96 468  24   0]
 [161 501  13   0]]


In [137]:
# import os
# from glob import glob
# from PIL import Image
# import numpy as np
# import pandas as pd

# GRAD_CAM_DIR = os.path.join(PROJECT_ROOT, "grad_cam_without_sp")
# images_path = glob(os.path.join(GRAD_CAM_DIR, "image", "*.png"))
# heatmaps_path = glob(os.path.join(GRAD_CAM_DIR, "heatmap", "*.png"))
# preds = pd.read_csv(os.path.join(GRAD_CAM_DIR, "predictions.csv"))
# preds_by_file = preds.set_index("filename")[["ground", "pred"]].to_dict("index")


# # sort both images and heatmaps
# images_path.sort()
# heatmaps_path.sort()

# labels = []
# images = []
# heatmaps = []
# indices = []
# predictions = []

# for idx, (img_path, hmap_path) in enumerate(zip(images_path, heatmaps_path)):
#     image_ = Image.open(img_path)
#     image_ = image_.convert("L")
#     image_ = np.array(image_)
#     images.append(image_)

#     heatmap_ = Image.open(hmap_path)
#     heatmap_ = heatmap_.convert("L")
#     heatmap_ = np.array(heatmap_)
#     heatmaps.append(heatmap_)

#     fname = os.path.basename(img_path)
#     hname = os.path.basename(hmap_path)
#     assert fname == hname, f"{fname} != {hname}"
#     assert fname in preds_by_file, f"{fname} not in predictions.csv"

#     row = preds_by_file[fname]
#     ground = int(row["ground"])
#     pred = int(row["pred"])
#     indices.append(idx)
#     labels.append(ground)
#     predictions.append(pred)


# images = np.array(images)
# heatmaps = np.array(heatmaps)
# labels = np.array(labels)
# predictions = np.array(predictions)


# data = np.stack([images, heatmaps], axis=-1)
# label_data = np.stack([indices, labels, predictions], axis=-1)
# os.makedirs(os.path.join(GRAD_CAM_DIR, "plot_data"), exist_ok=True)
# filenames = [os.path.basename(p) for p in images_path]
# np.save(os.path.join(GRAD_CAM_DIR, "plot_data", "filenames.npy"), filenames)
# np.save(os.path.join(GRAD_CAM_DIR, "plot_data", "grad_cam.npy"), data)
# np.save(os.path.join(GRAD_CAM_DIR, "plot_data", "labels_pred.npy"), label_data)


In [138]:
# import os
# import numpy as np
# import matplotlib.pyplot as plt

# # Use same config as above (PROJECT_ROOT from config cell)
# GRAD_CAM_DIR = os.path.join(PROJECT_ROOT, "grad_cam_without_sp")
# plot_data_dir = os.path.join(GRAD_CAM_DIR, "plot_data")

# # Load data
# data = np.load(os.path.join(plot_data_dir, "grad_cam.npy"))
# labels_data = np.load(os.path.join(plot_data_dir, "labels_pred.npy"))
# filenames = np.load(os.path.join(plot_data_dir, "filenames.npy"), allow_pickle=True)

# # Prepare output folder
# output_dir = os.path.join(GRAD_CAM_DIR, "gradcam_overlays")
# os.makedirs(output_dir, exist_ok=True)

# # Extract components
# images = data[:, :, :, 0]
# heatmaps = data[:, :, :, 1]

# # Loop through all entries using original filenames
# for image, heatmap, filename in zip(images, heatmaps, filenames):
#     fig, ax = plt.subplots(figsize=(1.5, 1.5), dpi=100)  # 150x150 pixels
#     ax.imshow(image, cmap='cubehelix_r')
#     levels = np.linspace(image.max() - 100, image.max(), 5)
#     ax.contour(heatmap, levels=levels, colors='k', alpha=0.5, linewidths=0.5)
#     ax.axis('off')

#     # Clean up filename and save
#     if isinstance(filename, bytes):
#         filename = filename.decode()

#     base_name = os.path.splitext(os.path.basename(filename))[0]
#     save_path = os.path.join(output_dir, f"{base_name}.png")

#     plt.subplots_adjust(left=0, right=1, top=1, bottom=0)  # Remove padding
#     plt.savefig(save_path)  # 1.5 inch * 100 dpi = 150 px
#     plt.close(fig)

# print(f" Saved 150x150 overlay images to '{output_dir}'.")
